# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2k24csaiml1e2411265-wq/ML_starter_FlyrankAI/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

### Action priority

1. **Review for refresh — `REFRESH_REVIEW`**: highest-priority items where the observed baseline/model signals suggest a useful content review. The reviewer checks whether the page is stale, whether the search intent has changed, and whether the observed CTR/position pattern is meaningful.
2. **Monitor — `MONITOR`**: items with weaker evidence. Keep them in the monitoring queue rather than spending immediate editorial effort.
3. **Hold / no action — `HOLD`**: items with insufficient evidence, missing important fields, or a signal that is likely explained by temporary search movement.

### Archetype → action mapping

| Observed archetype | Action | Reason code |
|---|---|---|
| Strong volume + weak CTR for position | Review title/snippet and search intent | `CTR_REVIEW` |
| Strong volume + unstable position | Review freshness and topical coverage | `REFRESH_REVIEW` |
| Low volume + weak evidence | Monitor; do not prioritize | `MONITOR` |
| Missing/insufficient data | Hold until evidence improves | `HOLD` |

### Decay / refresh insight

A decline signal is an observed change in search performance, not proof that content decay caused it. A refresh recommendation should therefore be treated as a review trigger. The reviewer should compare the page with current search intent and competing results before changing content.

In [1]:
import os, pathlib, pandas as pd, numpy as np

# The notebook can consume the ML-07 baseline queue if it exists.
queue_path='work/outputs/baseline_action_score.csv'
if os.path.exists(queue_path):
    queue=pd.read_csv(queue_path)
    print(f'Loaded baseline queue: {len(queue):,} rows')
else:
    # Safe fallback so the notebook remains executable before ML-07 is regenerated.
    queue=pd.DataFrame(columns=['rank','client_hash_id','content_hash_id','score','reason_code','action'])
    print('Baseline queue not found yet. Run ML-07 first to populate the ranked queue.')

# Standardize action labels for the playbook without changing the original score.
if len(queue):
    queue['playbook_action']=np.select([
        queue['reason_code'].eq('REFRESH_REVIEW'),
        queue['reason_code'].eq('MONITOR')],
        ['Review for refresh','Monitor'],default='Hold')
    display(queue.head(10))

Baseline queue not found yet. Run ML-07 first to populate the ranked queue.


## 2. Intended use and limits

**Who uses it:** a content strategist, SEO analyst, or editor uses the queue to decide which pages deserve human review first.

**What it is for:** prioritizing limited editorial attention using observed search-performance signals.

**Where it stops being valid:** the score is not a guarantee of traffic recovery, ranking improvement, or business value. It should not be interpreted causally. It is also not a replacement for page-quality review, search-intent analysis, or client-specific context.

**Cost/value thinking:** review the highest-priority items first because editorial time is limited. A high-volume page may have greater potential value, but expected value still depends on relevance, business importance, effort to update, and the likelihood that the observed issue is actionable.

In [2]:
if len(queue):
    queue['priority_band']=pd.cut(queue['score'],bins=[-np.inf,39,59,79,np.inf],labels=['Hold/low','Monitor','Review','High-priority'])
    summary=queue.groupby('priority_band',observed=False).size().reset_index(name='n')
    print('Priority-band counts:')
    display(summary)
    print('Highest observed score:',queue['score'].max())
    print('Median observed score:',queue['score'].median())
else:
    print('No queue statistics available until ML-07 output is regenerated.')

No queue statistics available until ML-07 output is regenerated.


## 3. Human review + the no-go list

### Required human checks before action

For every `Review for refresh` recommendation, a person should check:

- current search intent and whether the page still satisfies it;
- whether the content is factually accurate and useful;
- whether the observed CTR/position pattern could be caused by query mix, SERP features, seasonality, or temporary movement;
- whether the page has meaningful business/editorial value;
- whether the proposed change is proportionate to the evidence;
- whether a competing page or cannibalization issue changes the interpretation.

### No-go list — never automate

- Automatically publishing or deleting content based only on the score.
- Automatically changing titles, claims, prices, medical/legal/financial information, or other high-impact statements.
- Treating a model score as proof that a refresh will increase traffic.
- Automatically overriding an editor's decision.
- Using private client identifiers, private queries, or sensitive information in a public research output.

The playbook recommends **review**, not autonomous action.

In [3]:
review_rules=pd.DataFrame([
 {'check':'Search intent still matches','required':True},
 {'check':'Content is accurate and useful','required':True},
 {'check':'Temporary/seasonal effects considered','required':True},
 {'check':'Business/editorial value considered','required':True},
 {'check':'No automatic publishing/deletion','required':True},
 {'check':'No causal claim from score','required':True}
])
display(review_rules)

,check,required
0,Search intent still matches,True
1,Content is accurate and useful,True
2,Temporary/seasonal effects considered,True
3,Business/editorial value considered,True
4,No automatic publishing/deletion,True
5,No causal claim from score,True


## 4. Monitoring / retrain triggers

The playbook should be revisited when the data-generating process changes or when observed recommendation quality deteriorates.

**Light monitoring triggers:**
- feature distributions shift materially from the training/validation period;
- the decline rate changes substantially;
- missingness rises for important features;
- the share of `REFRESH_REVIEW` recommendations changes sharply;
- human reviewers frequently reject recommendations for the same reason;
- measured performance on a later, time-separated evaluation falls materially.

**Retrain trigger:** use a new time-separated evaluation window when enough new observations accumulate, or retrain earlier if a material data/schema/process change makes the existing feature relationships unreliable.

These are monitoring signals, not fixed production SLAs.

In [4]:
monitoring_triggers=pd.DataFrame([
 {'trigger':'Feature distribution shift','response':'Inspect feature definitions and consider retraining'},
 {'trigger':'Outcome/decline-rate shift','response':'Re-evaluate thresholds and validation'},
 {'trigger':'Feature missingness increase','response':'Investigate data pipeline and imputation'},
 {'trigger':'Sharp change in review-queue mix','response':'Audit rule/model and recent search changes'},
 {'trigger':'High human rejection rate','response':'Review reason codes and weak picks'},
 {'trigger':'Material drop on later time-separated test','response':'Revalidate and retrain if justified'}
])
display(monitoring_triggers)

,trigger,response
0,Feature distribution shift,Inspect feature definitions and consider retra...
1,Outcome/decline-rate shift,Re-evaluate thresholds and validation
2,Feature missingness increase,Investigate data pipeline and imputation
3,Sharp change in review-queue mix,Audit rule/model and recent search changes
4,High human rejection rate,Review reason codes and weak picks
5,Material drop on later time-separated test,Revalidate and retrain if justified


## 5. Exports for the paper

The paper needs a reproducible ranked queue and a small methodology receipt. The queue remains outside git by design; the notebook regenerates it.

In [5]:
pathlib.Path('work/outputs').mkdir(parents=True,exist_ok=True)

if len(queue):
    export_cols=[c for c in ['rank','client_hash_id','content_hash_id','score','reason_code','action','playbook_action','priority_band'] if c in queue.columns]
    playbook_queue=queue[export_cols].copy()
else:
    playbook_queue=pd.DataFrame(columns=['rank','content_hash_id','score','reason_code','action','playbook_action','priority_band'])

out_path='work/outputs/content_action_playbook_queue.csv'
playbook_queue.to_csv(out_path,index=False)

receipt={
    'lane':'Content Refresh & Performance Prediction',
    'purpose':'Human-reviewed content action prioritization',
    'automation_level':'Decision-support only',
    'primary_reason_code':'REFRESH_REVIEW',
    'no_go':'Autonomous publishing/deletion or causal interpretation',
    'queue_rows':int(len(playbook_queue))
}
import json
with open('work/outputs/action_playbook_receipt.json','w',encoding='utf-8') as f:
    json.dump(receipt,f,indent=2)

print('Exported:',out_path)
print('Exported: work/outputs/action_playbook_receipt.json')
display(playbook_queue.head(10))

Exported: work/outputs/content_action_playbook_queue.csv
Exported: work/outputs/action_playbook_receipt.json


,rank,content_hash_id,score,reason_code,action,playbook_action,priority_band


## Self-check

Before you submit, confirm each line honestly:

- [x] Ranked actions, reason codes, and archetype → action mapping are defined
- [x] Intended use, limits, human review, cost/value, and no-go cases are stated
- [x] Monitoring and retrain triggers are defined
- [x] Queue and receipt are exported to `work/outputs/`
- [x] Claims use observed, measured, directional, and decision-support language
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] Committed to my repo under `work/notebooks/w07_action_playbook.ipynb`